## Plotting the Gift-Eval Ablation Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean
import pprint

#### Configuration

In [ ]:
model_name = "Chronos 2"

#### Presets

In [ ]:
model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Chronos 2": "amazon-chronos-2",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
    "Moirai": "Salesforce-moirai-1.1-R-base_samples-100",
}

In [ ]:
apply_custom_style("../../config/plotting.yaml")
HOME_DIR = os.getenv("HOME")
root_dir = os.path.join(HOME_DIR, "tsfm-lens")  # type: ignore

## Load Data


In [ ]:
data_subdir_name = "all_components_rseed-99"

path_to_results = os.path.join(root_dir, "results", model_to_dir_mapping[model_name], data_subdir_name)

print(f"Loading original results from: {path_to_results}")
# list the files in the directory
pprint.pprint(f"Available files: {os.listdir(path_to_results)}")

In [ ]:
original_results_df = pd.read_csv(os.path.join(path_to_results, "original_all_results.csv"))

selected_layers = [i for i in range(12)]


ablated_results_df_dict = {
    layer_idx: pd.read_csv(
        os.path.join(
            path_to_results,
            f"head-mlp_layers_{layer_idx}_heads-all_all_results.csv",
        )
    )
    for layer_idx in selected_layers
}


In [ ]:
# validation: make sure the dataframe has 97 datasets
for layer_idx, ablated_results_df in ablated_results_df_dict.items():
    assert len(ablated_results_df) == 97, f"Expected 97 datasets, got {len(ablated_results_df)}"

In [ ]:
path_to_results

## Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True
is_normalized = False

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
seasonal_naive_df = pd.read_csv(os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv"))
print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

if NORMALIZE_BY_SEASONAL_NAIVE and not is_normalized:
    print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets. Normalizing...")
    original_results_df = normalize_by_seasonal_naive(original_results_df, seasonal_naive_df)

for layer_idx, ablated_results_df in ablated_results_df_dict.items():
    print(f"Layer {layer_idx}:")
    if NORMALIZE_BY_SEASONAL_NAIVE and not is_normalized:
        print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets. Normalizing...")
        ablated_results_df_dict[layer_idx] = normalize_by_seasonal_naive(ablated_results_df, seasonal_naive_df)
    else:
        print("Skipping normalization.")

is_normalized = True


### Display Available Metrics


In [ ]:
# Display available metrics
from typing import Any

metric_columns = [col for col in original_results_df.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate[Any](metric_columns, 1):
    print(f"{i}. {metric}")


In [ ]:
selected_metrics_lst = [
    "eval_metrics/MASE[0.5]",
    "eval_metrics/sMAPE[0.5]",
    "eval_metrics/NRMSE[mean]",
    "eval_metrics/MSIS",
    "eval_metrics/mean_weighted_sum_quantile_loss",
]


for layer_idx, ablated_results_df in ablated_results_df_dict.items():
    print(f"Layer {layer_idx}:")
    for sel_met in selected_metrics_lst:
        print(f"Selected metric: {sel_met}")
        geom_mean_original = gmean(original_results_df[sel_met])
        geom_mean_ablated = gmean(ablated_results_df[sel_met])
        percent_degradation = (geom_mean_ablated - geom_mean_original) / geom_mean_original * 100
        print(f"Percent degradation: {percent_degradation:.2f}%")


In [ ]:
gmean(original_results_df["eval_metrics/mean_weighted_sum_quantile_loss"])